# 8.3 權重能學到穩定風格嗎？

# 第 8 章：風格、個性與指令遵循

前置：第7章SFT。把風格與內容分開想：想像同一位老師可以說得生動，也可以精準遵循格式。本章先用人工編寫、內容固定的短回答校準目標；LoRA是後半支線。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：有個性也要可靠：三個分開的面向**

漂亮文筆不能替代正確答案或格式遵循。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/style.svg")))

**先想一想：**兩套資料題目完全不同，還能公平比較風格嗎？

要學穩定風格，資料中必須有穩定可辨的目標。用同一組題目、相同事實答案的兩套寫法，纔不會把知識差異誤當個性。

**把直覺寫成數學：**SFT目標仍是assistant-only CE；風格來自target內容，而不是單獨的「靈性參數」。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.data import render_chat

question = "2+2=?"
answers = {"簡短": "4", "生動": "4，就像兩雙筷子共有四根。"}
for style, answer in answers.items():
    x, y = render_chat([{"role": "user", "content": question}, {"role": "assistant", "content": answer}])
    print(style, "輸入長度", len(x), "有效答案tokens", int((y != -100).sum()))

**如何讀結果：**兩種長度使token預算不同；正式比較另記有效tokens，不把多寫字直接當進步。

**只改一件事：**只替換比喻，保持答案4與題目相同。
